# Notebook 02 · Una GAN que aprende una nube de puntos (2D)

**Introducción al Deep Learning — Módulo 3, Unidad 3: GANs (Parte I)**

Ampliamos la idea del Notebook 01 a **dos dimensiones**: la GAN aprenderá a generar puntos que caen sobre un
**anillo** (una distribución con forma). Es la misma mecánica adversaria, pero ahora el resultado se ve como
una nube de puntos.

1. La distribución real: puntos sobre un anillo.
2. Generador (ruido → punto 2D) y discriminador (punto 2D → real/falso).
3. Entrenar y comparar puntos reales vs. generados.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. La distribución real: un anillo de puntos

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
tf.random.set_seed(42); np.random.seed(42)

def datos_reales(n):
    ang = np.random.uniform(0, 2*np.pi, n)
    r = 1.0 + np.random.normal(0, 0.03, n)         # radio ~1 con algo de ruido
    return np.stack([r*np.cos(ang), r*np.sin(ang)], axis=1).astype("float32")

real = datos_reales(1000)
plt.figure(figsize=(4.5, 4.5))
plt.scatter(real[:, 0], real[:, 1], s=6, color="#4355FF")
plt.title("Distribución real (anillo)"); plt.axis("equal"); plt.tight_layout(); plt.show()

## 2. Generador y discriminador (para puntos 2D)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

DIM_RUIDO = 8
generador = Sequential([Input(shape=(DIM_RUIDO,)),
                        Dense(32, activation="relu"), Dense(32, activation="relu"),
                        Dense(2)])                              # salida: un punto (x, y)
discriminador = Sequential([Input(shape=(2,)),
                            Dense(32, activation="relu"), Dense(32, activation="relu"),
                            Dense(1, activation="sigmoid")])    # real / falso
bce = tf.keras.losses.BinaryCrossentropy()
opt_g = tf.keras.optimizers.Adam(0.0008); opt_d = tf.keras.optimizers.Adam(0.0008)
print("Modelos creados.")

## 3. Entrenar

In [ ]:
def ruido(n):
    return tf.random.normal((n, DIM_RUIDO))

@tf.function
def paso(reales):
    n = tf.shape(reales)[0]
    falsos = generador(ruido(n))
    with tf.GradientTape() as td:
        loss_d = bce(tf.ones_like(discriminador(reales)), discriminador(reales)) \
               + bce(tf.zeros_like(discriminador(falsos)), discriminador(falsos))
    opt_d.apply_gradients(zip(td.gradient(loss_d, discriminador.trainable_variables),
                             discriminador.trainable_variables))
    with tf.GradientTape() as tg:
        f = generador(ruido(n))
        loss_g = bce(tf.ones_like(discriminador(f)), discriminador(f))
    opt_g.apply_gradients(zip(tg.gradient(loss_g, generador.trainable_variables),
                             generador.trainable_variables))
    return loss_d, loss_g

for i in range(3000):
    ld, lg = paso(datos_reales(128))
    if i % 750 == 0:
        print(f"paso {i:4d} | loss_D {ld.numpy():.3f} | loss_G {lg.numpy():.3f}")
print("Entrenamiento terminado.")

## 4. Comparar puntos reales vs. generados

In [ ]:
gen = generador(ruido(1000)).numpy()
plt.figure(figsize=(5, 5))
plt.scatter(real[:, 0], real[:, 1], s=6, color="#4355FF", alpha=0.5, label="reales")
plt.scatter(gen[:, 0], gen[:, 1],  s=6, color="#FF647E", alpha=0.5, label="generados (GAN)")
plt.title("Anillo: reales vs. generados"); plt.axis("equal"); plt.legend(); plt.tight_layout(); plt.show()

Los puntos generados (rosa) se colocan sobre el anillo, imitando la forma de la distribución real. La
GAN ha aprendido **la estructura de los datos**, no solo un número.

## Experimenta tú

1. Cambia el anillo por dos grupos de puntos (dos gaussianas) y reentrena.
2. Sube o baja el número de pasos y observa cómo mejora la nube generada.
3. Cambia el tamaño de las redes.

## Resumen

- La misma mecánica adversaria funciona en 2D: el generador aprende a producir puntos con la forma correcta.
- El resultado se ve claramente como una **nube de puntos** que imita la distribución real.
- Entrenar una GAN es delicado: hay que equilibrar generador y discriminador.

➡️ En **GANs II** daremos el salto a **imágenes**: una DGAN con `Conv2DTranspose` y los tipos de GAN.